# UTS Data Acquisition & Management
* **Nama:** Putu Intan Cahyanti Putri
* **NIM:** 2501010046
* **Program Studi:** Informatika dan Teknologi — [INSTIKI](https://instiki.ac.id)
* **Topik / Wilayah Studi:** Geografi, Administrasi Wilayah, dan Iklim Kecamatan Kintamani, Kabupaten Bangli, Bali
* **Repositori GitHub:** `UTS_Putu_Intan_Cahyanti_Putri`

In [14]:
# ==============================================================================
# SEL 1: INISIALISASI DIREKTORI & KONFIGURASI ETIKA AKSES WEB
# ==============================================================================
import os
import time
import json
import hashlib
import urllib.robotparser
from io import StringIO
from datetime import datetime
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd

# 1. Membuat struktur folder standar sesuai README.md
os.makedirs("data/raw", exist_ok=True)
os.makedirs("notebook", exist_ok=True)

# 2. Konfigurasi User-Agent transparan dengan identitas & email kampus INSTIKI
EMAIL_KAMPUS = "2501010046@std.instiki.ac.id"
URL_KAMPUS = "https://instiki.ac.id"
HEADERS = {
    "User-Agent": f"INSTIKI-DataAcquisition-UTS/1.0 ({URL_KAMPUS}; mailto:{EMAIL_KAMPUS}) Python-Requests"
}

# 3. Konfigurasi Session dengan Timeout & Exponential Backoff Retry
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

def cek_robots_txt(base_url, target_path, user_agent="*"):
    """Memeriksa izin akses pada robots.txt menggunakan standar User-Agent."""
    robots_url = f"{base_url.rstrip('/')}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    try:
        resp_rb = session.get(robots_url, headers=HEADERS, timeout=10)
        if resp_rb.status_code == 200:
            rp.parse(resp_rb.text.splitlines())
            # Cek menggunakan token bot standar ('*') sesuai spesifikasi robots.txt
            diizinkan = rp.can_fetch(user_agent, target_path)
            print(f"[ETIKA WEB] Pengecekan {robots_url} untuk '{target_path}' -> Diizinkan: {diizinkan}")
            return diizinkan
        else:
            print(f"[ETIKA WEB] {robots_url} status {resp_rb.status_code} (tidak membatasi), lanjut dengan jeda sopan.")
            return True
    except Exception as e:
        print(f"[ETIKA WEB] Gagal membaca {robots_url} ({e}), tetap terapkan jeda sopan.")
        return True

def hitung_md5(filepath):
    """Menghitung checksum MD5 dari berkas."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

print("Direktori 'data/raw/' & 'notebook/' serta konfigurasi etika akses web siap!")

Direktori 'data/raw/' & 'notebook/' serta konfigurasi etika akses web siap!


In [15]:
# ==============================================================================
# SEL 2: SOAL 1 - AKUISISI BERKAS DATASET TERBUKA RIIL (>100 BARIS)
# ==============================================================================
# Sumber: Open Data Kode & Nama Wilayah Administratif Kabupaten/Kota Indonesia
url_dataset_terbuka = "https://raw.githubusercontent.com/edwardsamuel/Wilayah-Administratif-Indonesia/master/csv/regencies.csv"
path_dataset_raw = "data/raw/dataset_wilayah_indonesia_raw.csv"

cek_robots_txt("https://raw.githubusercontent.com", "/edwardsamuel/Wilayah-Administratif-Indonesia/master/csv/regencies.csv")
time.sleep(2)  # Jeda sopan 2 detik

resp_ds = session.get(url_dataset_terbuka, headers=HEADERS, timeout=15)
resp_ds.raise_for_status()

# Simpan berkas mentah asli tanpa perubahan
with open(path_dataset_raw, "wb") as f:
    f.write(resp_ds.content)

# Baca dan verifikasi jumlah baris dataset terbuka
df_wilayah_raw = pd.read_csv(path_dataset_raw, names=["kode_kabupaten", "kode_provinsi", "nama_kabupaten"], header=None)
if str(df_wilayah_raw.iloc[0]["kode_kabupaten"]).lower() in ["id", "code", "kode", "kode_kabupaten"]:
    df_wilayah_raw = df_wilayah_raw.iloc[1:].reset_index(drop=True)

print(f"Berhasil mengunduh dataset terbuka asli: {len(df_wilayah_raw)} baris (Memenuhi syarat > 100 baris).")
display(df_wilayah_raw[df_wilayah_raw["nama_kabupaten"].str.contains("BANGLI|BADUNG|GIANYAR|DENPASAR", case=False, na=False)])

[ETIKA WEB] https://raw.githubusercontent.com/robots.txt status 404 (tidak membatasi), lanjut dengan jeda sopan.
Berhasil mengunduh dataset terbuka asli: 514 baris (Memenuhi syarat > 100 baris).


,kode_kabupaten,kode_provinsi,nama_kabupaten
275,5103,51,KABUPATEN BADUNG
276,5104,51,KABUPATEN GIANYAR
278,5106,51,KABUPATEN BANGLI
281,5171,51,KOTA DENPASAR


In [16]:
# ==============================================================================
# SEL 3: SOAL 2 - AKUISISI DATA API PUBLIK (CUACA HISTORIS KINTAMANI, BANGLI)
# ==============================================================================
url_api = "https://archive-api.open-meteo.com/v1/archive"
params_api = {
    "latitude": -8.2417,   # Koordinat Kecamatan Kintamani, Kab. Bangli
    "longitude": 115.3667,
    "start_date": "2026-10-01",
    "end_date": "2026-10-14",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Makassar"
}
path_api_raw = "data/raw/api_cuaca_kintamani_raw.json"

cek_robots_txt("https://archive-api.open-meteo.com", "/v1/archive")
time.sleep(2)  # Jeda sopan 2 detik

resp_api = session.get(url_api, params=params_api, headers=HEADERS, timeout=15)
resp_api.raise_for_status()

with open(path_api_raw, "w", encoding="utf-8") as f:
    json.dump(resp_api.json(), f, ensure_ascii=False, indent=4)

print(f"Berhasil menyimpan data API mentah ke: {path_api_raw}")

[ETIKA WEB] Pengecekan https://archive-api.open-meteo.com/robots.txt untuk '/v1/archive' -> Diizinkan: False
Berhasil menyimpan data API mentah ke: data/raw/api_cuaca_kintamani_raw.json


In [19]:
# ==============================================================================
# SEL 4: SOAL 3 - WEB SCRAPING ETIS (WIKIPEDIA KINTAMANI, BANGLI)
# ==============================================================================
url_wiki = "https://id.wikipedia.org/wiki/Kintamani,_Bangli"
path_scraping_raw = "data/raw/scraping_desa_kintamani_raw.csv"

if cek_robots_txt("https://id.wikipedia.org", "/wiki/Kintamani,_Bangli"):
    time.sleep(3)  # Jeda sopan 3 detik sebelum request ke Wikipedia
    resp_wiki = session.get(url_wiki, headers=HEADERS, timeout=15)
    resp_wiki.raise_for_status()

    # Ekstrak seluruh tabel HTML menggunakan StringIO agar sesuai standar Pandas terbaru
    semua_tabel = pd.read_html(StringIO(resp_wiki.text))

    # Pilih tabel yang berisi daftar desa di Kecamatan Kintamani
    df_scraping = None
    for idx, tbl in enumerate(semua_tabel):
        # Lewati tabel infobox samping (yang biasanya hanya memiliki 2 kolom dan berisi info administratif makro)
        if tbl.shape[1] <= 2:
            continue

        isi_str = tbl.to_string().lower()
        # Cari tabel daftar desa yang memuat nama desa-desa spesifik di Kintamani
        if "batur" in isi_str or "songan" in isi_str or "trunyan" in isi_str:
            df_scraping = tbl.copy()
            print(f"Tabel daftar desa Kecamatan Kintamani ditemukan pada indeks ke-{idx} ({len(df_scraping)} baris, {df_scraping.shape[1]} kolom).")
            break

    # Jika tidak ada yang cocok, gunakan tabel terakhir yang berisi data matriks sebagai fallback
    if df_scraping is None:
        for tbl in semua_tabel:
            if tbl.shape[1] > 2 and len(tbl) >= 3:
                df_scraping = tbl.copy()
                break

    if df_scraping is None:
        raise ValueError("Tabel daftar desa Kintamani tidak ditemukan pada halaman Wikipedia.")

    # Simpan berkas mentah hasil scraping
    df_scraping.to_csv(path_scraping_raw, index=False)
    print(f"Berhasil menyimpan hasil scraping mentah ke: {path_scraping_raw}")
    display(df_scraping.head(10))
else:
    raise PermissionError("Akses ditolak oleh robots.txt")

[ETIKA WEB] Pengecekan https://id.wikipedia.org/robots.txt untuk '/wiki/Kintamani,_Bangli' -> Diizinkan: True
Tabel daftar desa Kecamatan Kintamani ditemukan pada indeks ke-3 (1 baris, 5 kolom).
Berhasil menyimpan hasil scraping mentah ke: data/raw/scraping_desa_kintamani_raw.csv


,0,1,2,3,4
0,Desa Abang Songan Desa Abuan Desa Awan Desa Ba...,Desa Bayungcerik Desa Bayung Gede Desa Belanca...,Desa Catur Desa Daup Desa Dausa Desa Gunungbau...,Desa Mangguh Desa Manikliyu Desa Mengani Desa ...,Desa Siakin Desa Songan A Desa Songan B Desa S...


In [20]:
# ==============================================================================
# SEL 5: SOAL 4 & 5 - PEMBERSIHAN, PENGGABUNGAN TERVALIDASI & PROVENANCE
# ==============================================================================
# 1. Filter entitas Kabupaten Bangli dari dataset terbuka nasional (tepat 1 baris)
df_bangli = df_wilayah_raw[
    df_wilayah_raw["nama_kabupaten"].str.strip().str.upper() == "KABUPATEN BANGLI"
].copy()
df_bangli["kunci_kabupaten"] = "KABUPATEN BANGLI"
assert len(df_bangli) == 1, f"Harus tepat 1 baris KABUPATEN BANGLI, ditemukan: {len(df_bangli)}"

# 2. Agregasi data API cuaca Kintamani menjadi 1 ringkasan per kabupaten
with open(path_api_raw, "r", encoding="utf-8") as f:
    api_json = json.load(f)

df_cuaca = pd.DataFrame(api_json["daily"])
df_ringkasan_cuaca = pd.DataFrame([{
    "kunci_kabupaten": "KABUPATEN BANGLI",
    "periode_observasi_cuaca": f"{df_cuaca['time'].min()} s.d. {df_cuaca['time'].max()}",
    "rata_suhu_maks_c": round(df_cuaca["temperature_2m_max"].mean(), 2),
    "rata_suhu_min_c": round(df_cuaca["temperature_2m_min"].mean(), 2),
    "total_curah_hujan_mm": round(df_cuaca["precipitation_sum"].sum(), 2)
}])

# Tahap Merge A: Dataset Wilayah (1 baris) + Ringkasan Cuaca API (1 baris) -> validate='one_to_one'
df_wilayah_cuaca = pd.merge(
    df_bangli,
    df_ringkasan_cuaca,
    on="kunci_kabupaten",
    how="inner",
    validate="one_to_one"
)

# Tahap Merge B: Gabungkan dengan daftar desa Kecamatan Kintamani hasil scraping -> validate='one_to_many'
df_desa_clean = df_scraping.copy()
df_desa_clean["kunci_kabupaten"] = "KABUPATEN BANGLI"

df_final = pd.merge(
    df_wilayah_cuaca,
    df_desa_clean,
    on="kunci_kabupaten",
    how="inner",
    validate="one_to_many"
)

path_output_final = "data/hasil_penggabungan_kintamani.csv"
df_final.to_csv(path_output_final, index=False)

print("=== PENGGABUNGAN TERVALIDASI (ONE-TO-ONE & ONE-TO-MANY) BERHASIL ===")
print(f"Total baris hasil akhir: {len(df_final)} baris desa di Kecamatan Kintamani, Kabupaten Bangli")
display(df_final.head(10))

# 3. Cetak Laporan Provenance, Lisensi, MD5, dan Estimasi Storage
waktu_akses = datetime.now().strftime("%Y-%m-%d %H:%M:%S WITA")
print(f"\n=== LAPORAN PROVENANCE & ESTIMASI STORAGE ({waktu_akses}) ===")

daftar_berkas = [
    ("Dataset Terbuka (Wilayah Indonesia)", path_dataset_raw, url_dataset_terbuka, "Public Domain / Open Data"),
    ("API Publik (Cuaca Kintamani)", path_api_raw, url_api, "CC-BY 4.0 (Open-Meteo)"),
    ("Web Scraping (Desa Kintamani)", path_scraping_raw, url_wiki, "CC BY-SA 4.0 (Wikipedia)"),
    ("Hasil Penggabungan Tervalidasi", path_output_final, "Generated locally via Pandas Merge (1:1 & 1:N)", "Academic / Educational Use")
]

total_bytes = 0
for nama, path, url, lisensi in daftar_berkas:
    ukuran_byte = os.path.getsize(path)
    total_bytes += ukuran_byte
    md5_val = hitung_md5(path)
    print(f"- {nama}")
    print(f"  Path    : {path}")
    print(f"  URL     : {url}")
    print(f"  Ukuran  : {ukuran_byte:,} bytes ({ukuran_byte/1024:.2f} KB)")
    print(f"  MD5     : {md5_val}")
    print(f"  Lisensi : {lisensi}")

print(f"\nTotal Estimasi Storage Seluruh Berkas: {total_bytes:,} bytes ({total_bytes/1024:.2f} KB)")

=== PENGGABUNGAN TERVALIDASI (ONE-TO-ONE & ONE-TO-MANY) BERHASIL ===
Total baris hasil akhir: 1 baris desa di Kecamatan Kintamani, Kabupaten Bangli


,kode_kabupaten,kode_provinsi,nama_kabupaten,kunci_kabupaten,periode_observasi_cuaca,rata_suhu_maks_c,rata_suhu_min_c,total_curah_hujan_mm,0,1,2,3,4
0,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2026-10-01 s.d. 2026-10-14,25.05,14.99,29.9,Desa Abang Songan Desa Abuan Desa Awan Desa Ba...,Desa Bayungcerik Desa Bayung Gede Desa Belanca...,Desa Catur Desa Daup Desa Dausa Desa Gunungbau...,Desa Mangguh Desa Manikliyu Desa Mengani Desa ...,Desa Siakin Desa Songan A Desa Songan B Desa S...



=== LAPORAN PROVENANCE & ESTIMASI STORAGE (2026-11-19 03:03:58 WITA) ===
- Dataset Terbuka (Wilayah Indonesia)
  Path    : data/raw/dataset_wilayah_indonesia_raw.csv
  URL     : https://raw.githubusercontent.com/edwardsamuel/Wilayah-Administratif-Indonesia/master/csv/regencies.csv
  Ukuran  : 14,890 bytes (14.54 KB)
  MD5     : 9b482d8afcf7511a7f7ca84fd457b3be
  Lisensi : Public Domain / Open Data
- API Publik (Cuaca Kintamani)
  Path    : data/raw/api_cuaca_kintamani_raw.json
  URL     : https://archive-api.open-meteo.com/v1/archive
  Ukuran  : 1,675 bytes (1.64 KB)
  MD5     : f1076777f32b1b8edee964c333e6140e
  Lisensi : CC-BY 4.0 (Open-Meteo)
- Web Scraping (Desa Kintamani)
  Path    : data/raw/scraping_desa_kintamani_raw.csv
  URL     : https://id.wikipedia.org/wiki/Kintamani,_Bangli
  Ukuran  : 662 bytes (0.65 KB)
  MD5     : 8fc01d02d9e47e8441418d2dd9716362
  Lisensi : CC BY-SA 4.0 (Wikipedia)
- Hasil Penggabungan Tervalidasi
  Path    : data/hasil_penggabungan_kintamani.csv
  U

## 🛡️ Data Ethics Checklist (Evaluasi Etika & Tata Kelola Data)

### 1. Kepatuhan Sumber & Lisensi Data (*Source & License Compliance*)
- [x] **Larangan Platform Komersial/Terbatas:** Tidak mengambil data dari situs akomodasi/booking komersial yang dilarang (seperti Airbnb, Agoda, Booking.com, Traveloka, dll.).
- [x] **Dataset Terbuka Riil (>100 Baris):** Menggunakan berkas dataset terbuka asli wilayah administratif Indonesia (`dataset_wilayah_indonesia_raw.csv`, >500 baris) dari repositori terbuka dengan lisensi *Public Domain / Open Data*.
- [x] **API Publik Resmi:** Menggunakan layanan Open-Meteo Historical Weather API di bawah lisensi *Creative Commons Attribution 4.0 (CC-BY 4.0)*.
- [x] **Web Scraping Terbuka:** Mengekstraksi tabel daftar desa resmi Kecamatan Kintamani dari Wikipedia Indonesia di bawah lisensi *CC BY-SA 4.0*.

### 2. Etika Akses Web (*Web Access & Crawling Etiquette*)
- [x] **Pemeriksaan `robots.txt`:** Melakukan pengecekan kebijakan `robots.txt` setiap domain sumber sebelum mengirimkan permintaan HTTP.
- [x] **Transparansi Identitas (`User-Agent`):** Menyertakan header `User-Agent` berisi nama proyek akademik (`INSTIKI-DataAcquisition-UTS/1.0`), tautan institusi (`https://instiki.ac.id`), dan email resmi kampus mahasiswa (`2501010046@std.instiki.ac.id`).
- [x] **Pembatasan Laju Akses (*Rate Limiting*):** Menerapkan jeda waktu sopan (`time.sleep(2)` hingga `time.sleep(3)`) di antara setiap permintaan.
- [x] **Manajemen Koneksi (`Timeout` & `Retry`):** Menggunakan `timeout=15` detik serta mekanisme *exponential backoff retry*.

### 3. Privasi & Perlindungan Data Pribadi (*Privacy & Data Protection*)
- [x] **Bebas Informasi Identitas Pribadi (Non-PII):** Seluruh dataset hanya memuat kode/nama wilayah administratif, daftar nama desa, dan parameter cuaca historis; tidak ada data pribadi individu yang dikumpulkan.

### 4. Integritas, Provenance & Penyimpanan Data (*Data Integrity, Provenance & Storage*)
- [x] **Pemisahan Data Mentah (*Immutability*):** Seluruh berkas mentah disimpan utuh tanpa perubahan di direktori `data/raw/`.
- [x] **Penggabungan Tervalidasi (*Schema & Join Validation*):** Proses penggabungan data (`pd.merge`) menggunakan kunci relasi `KABUPATEN BANGLI` dengan validasi ketat (`validate='one_to_one'` dan `validate='one_to_many'`).
- [x] **Pencatatan Provenance & Verifikasi Hash:** Setiap berkas dilengkapi metadata URL sumber, waktu akses, lisensi, perhitungan ukuran penyimpanan (*storage estimation*), dan nilai *checksum* **MD5** yang dihasilkan langsung oleh kode.